# 03 — From spiral winding to a Galactic clock

**Scientific question:** How can a velocity wave become a clock for a Galactic perturbation—and which parts of Lambert's clock are measured, reproduced, inferred, or model-dependent?

This notebook keeps four lanes separate:

1. **Physical mechanism demonstrator:** an educational Antoja-style toy, not a fit or N-body simulation.
2. **Released data:** Lambert Figure 9 and Figure 10 products from Zenodo record **18236902**.
3. **Forensic reproduction:** numerical fingerprints of Figure 10, not a documented author pipeline.
4. **Independent analysis:** one fully declared estimator we can reproduce from Figure 9 today.

> **Data boundary.** `Lz_Vr_fig9.fits` contains 70 binned summaries, not stars. `FFT_fig10.fits` contains 35 already-computed frequencies, powers, and power spreads—not raw Fourier inputs or Gaussian-fit parameters. The exact Figure-9 → Figure-10 author calculation is not publicly reproducible.

In [ ]:
from pathlib import Path

import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
from astropy.table import Table

from lambert_lab.data import (
    load_figure10_spectrum, load_figure9_wave, load_lambert_inventory,
)
from lambert_lab.dynamics import (
    PROJECT_SEED, angular_frequency, epicyclic_frequency, tidal_pattern_speed,
    toy_radial_velocity, transformed_angular_momentum,
    winding_time_uncertainty_flat_curve,
)
from lambert_lab.spectral import (
    legacy_frequency_grid, legacy_raw_power, local_maxima,
    lomb_scargle_on_grid, monte_carlo_periodogram, spectral_resolution,
    uniform_resample, window_normalized_periodogram,
)

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
VALIDATION_DIR = ROOT / 'figures' / 'validation'
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)
FREQUENCY_UNIT = u.kpc * u.km / u.s
inventory = load_lambert_inventory(ROOT)
print(f"Lambert release DOI: {inventory['release']['doi']}")
print(f"Cached archive SHA-256: {inventory['release']['archive']['sha256']}")

# Act 1 — How can one perturbation leave a clock?

### Question

Why should a radial-velocity pattern acquire more oscillations as time passes?

### Physical intuition

Imagine an impulsive, two-armed disturbance that initially gives neighboring guiding radii similar phases. For a flat rotation curve,

$$\Omega(R)=\frac{V_c}{R},\qquad \kappa(R)=\sqrt{2}\,\Omega(R),\qquad
\Omega_{\rm pattern}(R)=\Omega(R)-\frac{\kappa(R)}{2}.$$

The pattern rate varies with radius. Initially aligned phases therefore shear apart. At one observing azimuth, sampling different guiding radii intersects alternating phases, producing a signed $V_R$ wave. Longer elapsed time means more wraps and a higher-frequency wave in $R_g$ or, for a flat curve, $L_Z=R_gV_c$.

> ⚠️ **Mechanism demonstrator only.** We set an illustrative dimensionless amplitude and phase. The relative phase between the drawn arm curves and the illustrative $V_R$ sine wave is arbitrary: only the radius-dependent winding rate and increasing oscillation count are being demonstrated. This is not a fit to Figure 9, not an N-body calculation, has no self-gravity, and does not constrain Sagittarius or make one wrap correspond uniquely to one passage.

Notebook 2 showed the observed $R$–$V_\phi$ ridge; here we isolate the differential-winding ingredient that can produce an associated radial wave.

**Predict before revealing:** what should happen to the number of wave oscillations as the toy pattern winds longer?

In [ ]:
V_C_TOY = 239.26 * u.km / u.s
toy_radius = np.linspace(8, 22, 700) * u.kpc
toy_lz = (toy_radius * V_C_TOY).to(FREQUENCY_UNIT)
toy_times = [0.15, 0.45, 0.90] * u.Gyr

omega = angular_frequency(toy_radius, V_C_TOY)
kappa = epicyclic_frequency(omega, n=0)
pattern_speed = tidal_pattern_speed(omega, n=0)
assert u.allclose(kappa, np.sqrt(2) * omega)

fig = plt.figure(figsize=(12, 7.5))
grid = fig.add_gridspec(2, 3, height_ratios=[1.15, 1])
for column, elapsed in enumerate(toy_times):
    ax = fig.add_subplot(grid[0, column], projection='polar')
    phase = (pattern_speed * elapsed).to_value(u.one)
    for offset in [0, np.pi]:
        ax.plot(np.mod(phase + offset, 2*np.pi), toy_radius.value, lw=2)
    ax.set_title(f'{elapsed.value:.2f} Gyr')
    ax.set_yticklabels([])
    ax.grid(alpha=0.3)

ax_wave = fig.add_subplot(grid[1, :])
crossing_counts = []
for elapsed in toy_times:
    wave = toy_radial_velocity(
        toy_radius, elapsed, circular_speed=V_C_TOY, amplitude=1*u.one,
    ).value
    crossing_counts.append(np.count_nonzero(np.diff(np.signbit(wave))))
    ax_wave.plot(toy_lz.value, wave, label=f'{elapsed.value:.2f} Gyr')
ax_wave.axhline(0, color='0.5', lw=0.8)
ax_wave.set(
    xlabel=r'toy $L_Z=R_gV_c$ [kpc km s$^{-1}$]',
    ylabel='illustrative signed $V_R$ amplitude',
    title='At one observing azimuth: longer winding gives more oscillations',
)
ax_wave.legend(title='toy elapsed time', ncols=3)
fig.suptitle(r'Educational $m=2$ differential-winding demonstrator', y=1.01)
fig.tight_layout()
fig.savefig(VALIDATION_DIR / 'notebook03_toy_winding.png', dpi=140, bbox_inches='tight')
plt.show()
print('Zero-crossing diagnostic:', dict(zip(toy_times.value, crossing_counts)))
assert crossing_counts[2] > crossing_counts[1] > crossing_counts[0]

> 📈 **Result:** the radial dependence of $\Omega-\kappa/2$ winds the analytic two-arm phase and increases the wave's oscillation count. The displayed arm–wave phase offset is arbitrary, not a fitted physical phase.  
> 🧠 **Mechanism inference:** differential winding can encode elapsed time in spatial frequency.  
> ⚠️ **Not demonstrated:** that the Milky Way wave has this unique cause, that Sagittarius made it, or that the toy amplitude matches the Galaxy.

### 🧪 Try it yourself

Change only `TRY_TIME`. The amplitude, rotation curve, observing azimuth, and phase remain fixed; count how many sign changes appear.

In [ ]:
TRY_TIME = 0.60 * u.Gyr
try_wave = toy_radial_velocity(toy_radius, TRY_TIME, circular_speed=V_C_TOY)
try_wrap_indicator = np.count_nonzero(np.diff(np.signbit(try_wave.value)))
print(f'{TRY_TIME:.2f}: {try_wrap_indicator} zero crossings across the displayed L_Z support')

# Act 2 — What did Lambert actually release?

### Data / observable

The products below are checksum-validated members of Lambert et al. (2026), Zenodo DOI `10.5281/zenodo.18236902`:

- **Figure 9:** `Lz_Vr_fig9.fits`, HDU 1—70 rows of $L_Z$, a central $V_R$ statistic, and its uncertainty.
- **Figure 10:** `FFT_fig10.fits`, HDU 1—35 author-computed frequencies, powers, and one-sigma power spreads.

The Figure 9 wave is a released downstream measurement product, so we can replot it. We cannot rebuild its stellar selection or determine whether the central statistic is the caption's mean or the text's median. The published caption says mean, 70 bins, and 1400–4500; the body says median, 75 bins, and 2000–4500; arXiv v1 also prints an evident 45000 endpoint typo. The actual release is authoritative only for its 70 values and their 1500–4457.14 range. The FITS columns do not encode `TUNIT`; units used here are the column-label/paper interpretation recorded in the inventory.

In [ ]:
wave9 = load_figure9_wave(ROOT)
spectrum10 = load_figure10_spectrum(ROOT)
lz = wave9.lz.to_value(FREQUENCY_UNIT)
vr = wave9.vr.to_value(u.km/u.s)
vr_error = wave9.vr_uncertainty.to_value(u.km/u.s)
released_frequency = spectrum10.frequency.to_value(FREQUENCY_UNIT)

release_summary = Table(
    rows=[
        ('Figure 9', wave9.filename, wave9.hdu, len(lz), 'L_Z, V_R summary, V_R uncertainty'),
        ('Figure 10', spectrum10.filename, spectrum10.hdu, len(released_frequency), 'frequency, power, power spread'),
    ],
    names=['product', 'file', 'HDU', 'rows', 'released content'],
)
display(release_summary)
print(f'Figure 9 L_Z: {lz.min():.6f} to {lz.max():.6f}; spacing {np.diff(lz)[0]:.6f}')
print(f'Figure 10 ordering: {released_frequency[0]:.6f} down to {released_frequency[-1]:.6f}')

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.errorbar(lz, vr, yerr=vr_error, fmt='o-', ms=3, lw=1, capsize=2, color='tab:blue')
ax.axhline(0, color='0.45', lw=0.8)
ax.set(xlabel=r'$L_Z$ [kpc km s$^{-1}$]', ylabel=r'released $V_R$ statistic [km s$^{-1}$]',
       title='Lambert Figure 9 released binned wave (70 summaries, not stars)')
fig.tight_layout()
fig.savefig(VALIDATION_DIR / 'notebook03_figure9_wave.png', dpi=140, bbox_inches='tight')
plt.show()

### Why transform the coordinate before an FFT?

The Antoja power-law model uses

$$x=L_Z^{(n-1)/(n+1)}.$$

The point of this re-expression is dynamical: in the idealized model it makes winding features approximately equally spaced in $x$. For the flat-curve baseline $n=0$,

$$x=1/L_Z.$$

$L_Z$ has units kpc km s$^{-1}$, so $x$ has reciprocal angular-momentum units. Fourier frequency $f$ is reciprocal to $x$, hence has angular-momentum units. A frequency and a spacing are different objects:

$$\Delta x=\frac{1}{f}.$$

**Predict before calculating:** are uniform $L_Z$ bins still uniform after taking $1/L_Z$?

In [ ]:
x = transformed_angular_momentum(wave9.lz, n=0).to_value(1/FREQUENCY_UNIT)
dlz = np.diff(lz)
dx = np.diff(x)
print(f'L_Z spacing min/max: {dlz.min():.8f}, {dlz.max():.8f}')
print(f'x=1/L_Z spacing magnitude min/max: {np.abs(dx).min():.3e}, {np.abs(dx).max():.3e}')
print(f'ratio of largest to smallest |Delta x|: {np.abs(dx).max()/np.abs(dx).min():.2f}')
assert np.allclose(dlz, dlz[0], rtol=1e-12)
assert not np.allclose(dx, dx[0], rtol=1e-3, atol=0)

# Act 3 — Forensics: what appears to have produced Figure 10?

This lane asks about computational lineage, not preferred methodology.

### A forensic frequency-grid fingerprint

If we take only the first two transformed samples and define

$$d_{\rm legacy}=x_1-x_0,$$

then use the ordinary 70-point FFT frequency convention, every released Figure 10 frequency is recovered. This is striking because the full $x$ series is not uniformly spaced. We derive the result below rather than hard-coding it.

In [ ]:
legacy_frequency = legacy_frequency_grid(x)
grid_residual = legacy_frequency - released_frequency
np.testing.assert_allclose(legacy_frequency, released_frequency, rtol=1e-12, atol=1e-9)
print(f'd_legacy = {x[1]-x[0]:.12e} [(kpc km/s)^-1]')
print(f'max |forensic grid - released grid| = {np.max(np.abs(grid_residual)):.3e} kpc km/s')
print('forensic frequency-grid fingerprint: YES')

Next we apply the corresponding unnormalized $|\mathrm{FFT}(V_R)|^2$ calculation to the released central values. For visualization only, each power curve is divided by its own maximum; the released spread is divided by the released-power maximum. No preprocessing parameter is tuned, and absolute normalization is not claimed.

In [ ]:
legacy_power = legacy_raw_power(vr)
legacy_correlation = float(np.corrcoef(legacy_power, spectrum10.power)[0, 1])
released_scaled = spectrum10.power / np.max(spectrum10.power)
legacy_scaled = legacy_power / np.max(legacy_power)
floor_scaled = (spectrum10.power - legacy_power * np.dot(spectrum10.power, legacy_power)
                / np.dot(legacy_power, legacy_power))
print(f'Pearson shape correlation = {legacy_correlation:.8f}')
print(f'Residual median after a no-intercept scale comparison = {np.median(floor_scaled):.1f} released-power units')

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].hist(np.abs(dx), bins=14, color='0.35')
axes[0].axvline(abs(x[1]-x[0]), color='tab:red', ls='--', label=r'$|d_{legacy}|$')
axes[0].set(xlabel=r'$|\Delta x|$ [(kpc km s$^{-1}$)$^{-1}$]', ylabel='adjacent pairs',
            title=r'Nonuniform spacing after $x=1/L_Z$')
axes[0].legend()
axes[1].plot(released_frequency, released_scaled, 'o-', ms=4, label='released Figure 10 power / max')
axes[1].fill_between(released_frequency,
                     (spectrum10.power-spectrum10.power_spread)/np.max(spectrum10.power),
                     (spectrum10.power+spectrum10.power_spread)/np.max(spectrum10.power),
                     color='tab:blue', alpha=0.12, label='released 1-sigma spread / same max')
axes[1].plot(legacy_frequency, legacy_scaled, '.--', label=r'raw legacy $|FFT(V_R)|^2$ / max')
axes[1].set(xlabel=r'frequency [kpc km s$^{-1}$]', ylabel='shape-normalized power',
            title=f'Forensic shape fingerprint: r={legacy_correlation:.5f}')
axes[1].legend(fontsize=8)
fig.tight_layout()
fig.savefig(VALIDATION_DIR / 'notebook03_forensic_fft.png', dpi=140, bbox_inches='tight')
plt.show()

> 📈 **Forensic result:** the frequency grid matches to numerical precision and the raw-power shape correlation is about 0.99986. A roughly positive residual floor is an observation, not proof of a particular Monte Carlo distribution or aggregation rule.  
> ⚠️ **Methodological problem:** an ordinary FFT assumes equal independent-coordinate spacing; $1/L_Z$ is not equally spaced here. The near-match is evidence about computational lineage, not validation of this estimator.  
> ⚠️ **Still unknown:** mean versus median aggregation, draw distribution, one-sigma definition, detrending, windowing, padding, normalization, Gaussian-fit windows/weights, and random seed. The public predecessor notebook is historical evidence only, not final executable provenance.

# Act 4 — An independent analysis we can actually define

Our baseline is deliberately separate from the forensic lane:

1. Compute $x=1/L_Z$ and sort it ascending.
2. Linearly interpolate onto exactly 70 uniform points covering only the observed support.
3. For each of 1000 realizations, draw independent Gaussians at the released points using the released $V_R$ uncertainties. Independence and Gaussianity are **our assumptions**.
4. Interpolate each draw, subtract its mean, apply a Hann window, and take a one-sided FFT.
5. Report $|\mathrm{FFT}|^2/\sum w_i^2$ as **window-normalized relative power**, not a physical PSD.
6. Summarize with our median and 16th–84th percentiles using fixed seed **20260920**.

> **Uncertainty scope.** We treat the released Figure-9 uncertainties as independent Gaussian scales in our analysis. Covariance between the released Figure-9 bins is not publicly available, while interpolation onto a uniform $x$ grid creates correlated resampled values by construction. The displayed MC 68% band therefore propagates the released per-bin uncertainty under our declared assumptions; it is not the complete uncertainty of the original analysis.

**Predict before revealing:** would plotting a denser frequency grid necessarily create more physical resolving power?

In [ ]:
mc = monte_carlo_periodogram(
    x, vr, vr_error, realizations=1000, seed=PROJECT_SEED, window='hann',
)
published_frequencies = np.array([1313.0, 5878.6])
published_widths = np.array([477.2, 1471.9])
x_span, baseline_resolution, cycles = spectral_resolution(x, published_frequencies)
first_fft_bin_frequency = mc.frequency[0]
first_fft_bin_relative_power = mc.central_power[0] / mc.central_power.max()
peak_indices = local_maxima(mc.central_power, strongest=5)
peak_table = Table(
    [mc.frequency[peak_indices], mc.central_power[peak_indices] / mc.central_power.max()],
    names=['independent FFT interior local-maximum frequency', 'power / global maximum'],
)
print(f'x span = {x_span:.12e} [(kpc km/s)^-1]')
print(f'baseline-limited frequency scale 1/x_span = {baseline_resolution:.3f} kpc km/s')
print(f'numerical FFT-bin spacing = {np.diff(mc.frequency)[0]:.3f} kpc km/s')
print(f'first positive / boundary FFT bin = {first_fft_bin_frequency:.3f} kpc km/s '
      f'(relative power {first_fft_bin_relative_power:.3f})')
display(peak_table)
display(Table(
    [published_frequencies, cycles],
    names=['Lambert published fitted frequency', 'cycles across released support'],
))

The numerical grid spacing and physical resolving power are related but not identical. Here $1/x_{\rm span}\approx2261$ kpc km s$^{-1}$ is the characteristic baseline-limited scale. The 1313 feature spans only about 0.58 cycle, while the 5878.6 feature spans about 2.60 cycles. A denser evaluation grid cannot manufacture additional independent information. In particular, a sub-cycle feature is intrinsically difficult to localize from this released table alone. This limitation applies to **our reconstruction from Figure 9 support**; it does not prove that an unreleased source-level analysis was invalid.

In [ ]:
ls_frequency = np.linspace(released_frequency.min(), released_frequency.max(), 4000)
ls_unweighted_power = lomb_scargle_on_grid(x, vr, None, ls_frequency)
ls_weighted_power = lomb_scargle_on_grid(x, vr, vr_error, ls_frequency)
ls_unweighted_strongest = ls_frequency[np.argmax(ls_unweighted_power)]
ls_weighted_strongest = ls_frequency[np.argmax(ls_weighted_power)]
display(Table(
    rows=[
        ('unweighted native-x Lomb-Scargle', ls_unweighted_strongest),
        ('uncertainty-weighted native-x Lomb-Scargle', ls_weighted_strongest),
    ],
    names=['estimator', 'strongest frequency [kpc km/s]'],
))

fig, ax = plt.subplots(figsize=(10, 5.2))
central_scaled = mc.central_power / mc.central_power.max()
median_scaled = mc.median_power / mc.median_power.max()
scale = mc.median_power.max()
ax.plot(mc.frequency, central_scaled, 'o-', ms=4, label='central values: uniform-x Hann FFT')
ax.plot(mc.frequency, median_scaled, '-', lw=2, label='our 1000-draw MC median')
ax.fill_between(mc.frequency, mc.lower_power/scale, mc.upper_power/scale,
                color='tab:blue', alpha=0.18, label='our central 68% interval')
ax.plot(ls_frequency, ls_unweighted_power/ls_unweighted_power.max(),
        color='tab:green', lw=1.5, label='unweighted Lomb-Scargle on native nonuniform x')
ax.plot(ls_frequency, ls_weighted_power/ls_weighted_power.max(), color='tab:orange', lw=1.5,
        label='weighted Lomb-Scargle on native nonuniform x')
for frequency, width in zip(published_frequencies, published_widths):
    ax.axvline(frequency, color='0.25', ls='--', lw=1)
    ax.axvspan(frequency-width, frequency+width, color='0.5', alpha=0.08)
ax.set(xlim=(0, released_frequency.max()), xlabel=r'frequency [kpc km s$^{-1}$]',
       ylabel='within-method relative power',
       title='Independent spectra; dashed lines are Lambert published Gaussian-fit frequencies')
ax.legend(fontsize=8, ncols=2)
fig.tight_layout()
fig.savefig(VALIDATION_DIR / 'notebook03_independent_spectra.png', dpi=140, bbox_inches='tight')
plt.show()

On the exact same explicit frequency grid, the strongest unweighted native-$x$ Lomb–Scargle location is about **6215** kpc km s$^{-1}$, versus about **5887** kpc km s$^{-1}$ when uncertainty weighted. Thus the broad $\sim5.9\text{–}6.2\times10^3$ neighbourhood persists without uncertainty weighting, while its maximum shifts by about 328 kpc km s$^{-1}$. The weighted calculation changes not only the handling of nonuniform sampling but also the sample weighting and floating-mean solution; it is therefore a robustness check, not a controlled one-variable sampling experiment. We retain the displacement rather than tuning it away.

The uniform-$x$ Hann FFT samples much more coarsely. Its first positive **boundary bin** is near 2.23×10³ kpc km s$^{-1}$ and is globally dominant; it is not an interior local peak. The interior local maxima are led by the feature near 6.69×10³, followed by one near 1.11×10⁴. Keeping the dominant boundary bin visible exposes the baseline-limited low-frequency behaviour. We report what the declared algorithms produce; we do not use Lambert's peaks to tune preprocessing or force exactly two maxima. Absolute Lomb–Scargle and FFT powers are not compared because their normalizations differ.

### Compact sensitivity experiments

We now change one declared choice at a time: signal ($V_R$ or $V_R/L_Z$), leakage control (rectangular or Hann), sampling estimator (legacy, uniform-$x$, or Lomb–Scargle), and power-law slope. For $n\ne0$ we rebuild $x=L_Z^{(n-1)/(n+1)}$ and its spectrum. Because the dimensional convention changes with the exponent, we compare normalized spectral structures and do not invent an $n\ne0$ timing-unit conversion here.

In [ ]:
sensitivity_rows = []
for signal_name, signal in [('V_R', vr), ('V_R/L_Z', vr/lz)]:
    grid, resampled = uniform_resample(x, signal, sample_count=70)
    for window in ['rectangular', 'hann']:
        frequency, power = window_normalized_periodogram(grid, resampled, window=window)
        indices = local_maxima(power, strongest=3)
        sensitivity_rows.append((signal_name, window, ', '.join(f'{frequency[i]:.0f}' for i in indices)))
display(Table(rows=sensitivity_rows, names=['signal', 'window', 'three strongest local maxima [kpc km/s]']))

slope_rows = []
for n_value in [0.0, -0.03, -0.06, -0.10]:
    x_n = transformed_angular_momentum(wave9.lz, n=n_value).value
    grid_n, signal_n = uniform_resample(x_n, vr, sample_count=70)
    frequency_n, power_n = window_normalized_periodogram(grid_n, signal_n, window='hann')
    indices_n = local_maxima(power_n, strongest=3)
    slope_rows.append((n_value, np.ptp(x_n), ', '.join(f'{frequency_n[i]:.3g}' for i in indices_n)))
display(Table(rows=slope_rows, names=['n', 'transformed-coordinate span', 'three strongest local maxima [native reciprocal-x units]']))

The dominant broad high-frequency location persists near 6–7×10³ for the $n=0$ uniform-$x$ choices, but leakage control changes secondary maxima and relative prominence. $V_R/L_Z$ changes amplitudes more than the leading interior local-maximum location. The weighted and unweighted Lomb–Scargle strongest locations above test whether the same frequency neighbourhood persists without uncertainty weighting; their difference is part of the result. Changing $n$ changes the coordinate's scale and the spectrum itself, so raw frequency numbers across slopes are not directly comparable without their native units. Lambert's printed sensitivity list includes conspicuous **$n=-0.5$** alongside other values; we preserve it as an unresolved manuscript value and do not silently replace it. Antoja's documented sequence is $0,-0.03,-0.06,-0.1$.

# Act 5 — When does frequency become time?

Timing is a separate, model-dependent map. We use Lambert's **published Gaussian-fit frequencies**, not peaks selected from our independent spectra:

$$f_1=1313.0\pm477.2,\qquad f_2=5878.6\pm1471.9.$$

With Lambert/Antoja's $n=0$, $V_0=239.26$ km s$^{-1}$, and $R_0=8.277$ kpc, the logical chain is

$$f\quad\longrightarrow\quad\Delta x=1/f\quad\longrightarrow\quad
t=\frac{\pi}{(1-\sqrt{2}/2)V_0^2\Delta x}.$$

$R_0$ cancels algebraically for $n=0$, but we record the adopted value because it belongs to the published model specification. The frequency widths are propagated directly through the linear $t\propto f$ relation; they are published Gaussian widths, not posterior uncertainties reconstructed here.

In [ ]:
V0 = 239.26 * u.km/u.s
R0 = 8.277 * u.kpc
timing_rows = []
times = []
time_errors = []
for frequency, width in zip(published_frequencies, published_widths):
    f_quantity = frequency * FREQUENCY_UNIT
    delta_x = (1/f_quantity).to(1/FREQUENCY_UNIT)
    time, uncertainty = winding_time_uncertainty_flat_curve(
        f_quantity, width * FREQUENCY_UNIT, circular_speed=V0,
    )
    times.append(time.to_value(u.Gyr)); time_errors.append(uncertainty.to_value(u.Gyr))
    timing_rows.append((frequency, delta_x.value, time.value, uncertainty.value))
display(Table(
    rows=timing_rows,
    names=['published f [kpc km/s]', 'Delta x [(kpc km/s)^-1]', 'computed t [Gyr]', 'direct sigma_t [Gyr]'],
))
print(f'Adopted n=0, V0={V0}, R0={R0}; direct rounded results: '
      f'{times[0]:.2f} +/- {time_errors[0]:.2f} Gyr and '
      f'{times[1]:.2f} +/- {time_errors[1]:.2f} Gyr')

fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.errorbar([1, 2], times, yerr=time_errors, fmt='o', ms=8, capsize=5, color='tab:purple')
ax.set(xticks=[1, 2], xticklabels=[r'$f_1=1313$', r'$f_2=5878.6$'],
       ylabel='model-mapped winding time [Gyr]', xlim=(0.5, 2.5),
       title='Published frequencies mapped through the n=0 tidal-winding model')
ax.text(2.08, 0.72, 'paper also prints ±0.23 Gyr', ha='center', fontsize=9)
fig.tight_layout()
fig.savefig(VALIDATION_DIR / 'notebook03_timing_conversion.png', dpi=140, bbox_inches='tight')
plt.show()

> 📈 **Reproducible calculation:** the exact constants give 0.2406±0.0874 and 1.0770±0.2697 Gyr before presentation rounding, consistent with Lambert's approximately **0.25±0.09** and **1.10±0.28 Gyr**.  
> ⚠️ **Visible inconsistency:** Lambert elsewhere prints **1.10±0.23 Gyr**. Direct propagation of the quoted 1471.9 frequency width supports approximately 0.27–0.28 Gyr. The 0.23/0.28 provenance discrepancy remains unresolved; we do not choose silently.  
> ⚠️ **Model dependence:** frequency becomes elapsed time only under the adopted power-law, epicyclic, tidally winding spiral picture.

# Act 6 — What have we actually learned?

## Reproducibility ledger

| Stage | Input | What we recompute | Status | Limitation |
|---|---|---|---|---|
| Figure 9 wave | `Lz_Vr_fig9.fits`, HDU 1 | plot 70 released summaries/errors | **YES** | not stars; mean/median, range, 70/75-bin provenance conflict |
| transformed $x$ | released $L_Z$ | $x=1/L_Z$ with units | **YES** | model-motivated coordinate |
| Figure-10 legacy frequency grid | first two transformed samples | ordinary 70-point FFT convention | **YES (fingerprint)** | not a documented or preferred procedure |
| released power-shape fingerprint | released central $V_R$ | raw unnormalized $|FFT|^2$ shape | **PARTIAL** | absolute normalization and floor unexplained |
| exact Lambert MC spectrum | Figure 9 plus undocumented choices | none claimed | **NO** | draw, aggregation, band, preprocessing, fit, seed absent |
| independent uniform-$x$ spectrum | Figure 9 | declared linear/Hann/1000-draw analysis | **YES** | our assumptions; finite-support resolution |
| Lomb–Scargle robustness | native nonuniform $x$ | weighted and unweighted spectra on the same explicit grid | **YES** | weighting and floating-mean solution differ; compare locations, not absolute FFT power |
| published peak fits | paper values | overlay only | **NO** | fit samples/windows/weights absent |
| frequency-to-time conversion | published fitted frequencies/equation | $f\to\Delta x\to t$ | **YES** | model-dependent; 0.23/0.28 conflict |

## Causal-inference ladder

1. **RELEASED / directly observed downstream product:** a radial-velocity wave exists in the released $L_Z$–$V_R$ summary.
2. **NUMERICAL / derived:** the wave contains spectral structure; exact structure depends on sampling and leakage choices.
3. **MODEL-DEPENDENT INFERENCE:** under the Antoja tidally winding spiral model, transformed-coordinate spacing maps to elapsed perturbation time.
4. **FURTHER PHYSICAL INTERPRETATION:** those times can be compared with proposed Sagittarius passages.
5. **NOT ESTABLISHED:** Sagittarius as the perturber, exactly two physical impacts, uniqueness of the tidal interpretation, or a fully self-consistent Galactic response.

Complications include bar/secular structure, self-gravity, overlapping perturbations, a non-power-law potential, an imperfect transformation, and finite sampling/binning. A spectrum can support a clock calculation without uniquely identifying the clockmaker.

> **The wave is measured; the spectrum is derived; the clock belongs to a model; the identification with Sagittarius is another inference.**

## Methods and provenance notes

- Lambert et al. (2026), *The Astronomical Journal* 171:292, DOI `10.3847/1538-3881/ae5100`.
- Lambert Figure Data, Zenodo record 18236902, DOI `10.5281/zenodo.18236902`; exact file/HDU/checksum provenance is loaded from `data/data_inventory.json`.
- Antoja et al. (2022), “Tidally induced spiral arm wraps encoded in phase space,” for the power-law transformation and winding interpretation.
- `docs/notebook3_reproducibility_audit.md`, the local methodological handoff separating released, forensic, independent, and model-dependent stages.
- Lomb (1976), Scargle (1982), and Astropy `timeseries.LombScargle` documentation support the nonuniform-sampling frequency-location check.

No code or data are fetched at runtime. No public precursor repository is treated as the final Lambert pipeline.

In [ ]:
NOTEBOOK_STAGE = 'task-5-complete'
assert NOTEBOOK_STAGE == 'task-5-complete'
assert legacy_correlation > 0.999
assert np.all(np.isfinite(ls_unweighted_power))
assert np.all(np.isfinite(ls_weighted_power))
assert len(mc.median_power) == 35
print('Notebook 3 completed offline with fixed seed', PROJECT_SEED)